# Avance 4: Arquitectura en la Nube (AWS)
**Objetivo:** Diseñar una arquitectura serverless escalable en AWS para soportar las operaciones en tiempo real y el flujo analítico de FleetLogix, minimizando costos operativos.

## Diagrama de Arquitectura
*(Asegúrate de que el archivo `aws_architecture_diagram.png` esté en la misma carpeta para que se visualice correctamente).*

![Arquitectura AWS FleetLogix](aws_architecture_diagram.png)

**Flujo Principal:** La App móvil envía datos a través de **API Gateway**. Las funciones **Lambda** procesan esta información, guardando el estado en tiempo real en **DynamoDB** y almacenando un histórico pesado en **Amazon S3**. Finalmente, S3 alimenta el Data Warehouse en **Snowflake** para analítica, mientras **CloudWatch** monitorea todo el sistema.

 código Extractor  extractor_mongo.py

In [ ]:
import json
import boto3
import pandas as pd
import os
from pymongo import MongoClient
from datetime import datetime

def lambda_handler(event, context):
    try:
        print("Iniciando extracción de telemetría desde MongoDB Atlas...")
        
        # 1. Conexión a MongoDB
        mongo_uri = os.environ['MONGO_URI']
        client = MongoClient(mongo_uri)
        db = client['fleetlogix_iot']
        collection = db['telemetria_gps']
        
        # 2. Extracción
        cursor = collection.find({})
        df_mongo = pd.DataFrame(list(cursor))
        
        # 3. Transformación básica y subida
        if not df_mongo.empty:
            df_final = df_mongo[['vehicle_id', 'timestamp', 'status']] # Columnas de ejemplo
            
            batch_id = int(datetime.now().timestamp())
            file_name = f"/tmp/telemetria_{batch_id}.csv"
            df_final.to_csv(file_name, index=False)
            
            s3 = boto3.client('s3')
            bucket_name = os.environ['S3_BUCKET_NAME']
            s3_key = f"raw/iot/telemetria_{batch_id}.csv"
            
            s3.upload_file(file_name, bucket_name, s3_key)
            
            return {
                'statusCode': 200,
                'body': json.dumps(f'Éxito: Telemetría subida a {s3_key}')
            }
        else:
            return {'statusCode': 204, 'body': 'No se encontraron datos nuevos.'}
            
    except Exception as e:
        print(f"Error: {e}")
        return {
            'statusCode': 500,
            'body': json.dumps(f'Error en la ejecución: {str(e)}')
        }

 código Extractor  extractor_postgres.py

In [ ]:
import json
import boto3
import psycopg2
import pandas as pd
import os
from datetime import datetime

# En Lambda, las credenciales se cargan desde "Variables de Entorno" de la propia consola de AWS
def lambda_handler(event, context):
    try:
        print("Iniciando extracción desde PostgreSQL...")
        
        # 1. Conexión a Postgres
        conn = psycopg2.connect(
            host=os.environ['PG_HOST'],
            database=os.environ['PG_DATABASE'],
            user=os.environ['PG_USER'],
            password=os.environ['PG_PASSWORD'],
            port=os.environ['PG_PORT']
        )
        
        # 2. Extracción de datos
        query = "SELECT * FROM sales_data LIMIT 1000;" # Ejemplo simplificado
        df = pd.read_sql(query, conn)
        conn.close()
        
        # 3. Guardado temporal y subida a S3
        batch_id = int(datetime.now().timestamp())
        file_name = f"/tmp/postgres_data_{batch_id}.csv" # En Lambda solo se puede escribir en /tmp
        df.to_csv(file_name, index=False)
        
        s3 = boto3.client('s3')
        bucket_name = os.environ['S3_BUCKET_NAME']
        s3_key = f"raw/postgres/data_{batch_id}.csv"
        
        s3.upload_file(file_name, bucket_name, s3_key)
        
        return {
            'statusCode': 200,
            'body': json.dumps(f'Éxito: Datos subidos a S3 en {s3_key}')
        }
        
    except Exception as e:
        print(f"Error: {e}")
        return {
            'statusCode': 500,
            'body': json.dumps(f'Error en la ejecución: {str(e)}')
        }